# Galaxy Merger Tracking with simbanator

This notebook walks through the full merger-detection pipeline:

1. **Setup** – imports and parameters  
2. **Data format** – what the pipeline expects  
3. **Mock data** – self-contained test run (no real catalogs needed)  
4. **Run `process_galaxies_with_tracks`** – find companions in each snapshot  
5. **Explore results** – inspect `Galaxy` and `Progenitor` objects  
6. **`analyze_mergers`** – classify into major / minor counts  
7. **Visualisation** – merger rate, per-galaxy history, mass-ratio distribution  
8. **Real-data template** – drop-in cell for actual SIMBA catalogs  

## 1 · Setup

In [ ]:
import os
import tempfile
import h5py
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits

from simbanator.analysis.mergers import (
    Galaxy,
    Progenitor,
    process_galaxies_with_tracks,
    analyze_mergers,
)
from simbanator.io.paths import OutputPaths

plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})

In [ ]:
# Ensure the source tree is found before any installed copy in site-packages.
# Permanent fix: run once in your cluster environment:
#   pip install -e /home/lorenzong/analize_simba_cgm
import sys
_pkg_root = '/home/lorenzong/analize_simba_cgm'
if _pkg_root not in sys.path:
    sys.path.insert(0, _pkg_root)

### Parameters

| Parameter | Meaning |
|---|---|
| `BOX_SIZE` | Periodic box side length **in position units** (Mpc/h) |
| `SEARCH_RADIUS_FACTOR` | Search sphere = this × stellar half-mass radius |
| `MASS_THRESHOLD` | Minimum neighbour stellar mass (M☉) to consider |
| `RHALF_UNIT_FACTOR` | Converts `radii_stellar_half_mass` to position units (1e-3 = kpc/h → Mpc/h) |
| `MASS_RATIO_MAJOR` | Mass ratio ≥ this → major merger |
| `MASS_RATIO_MINOR` | Mass ratio ≥ this (and < major) → minor merger |

In [ ]:
BOX_SIZE            = 100.0   # Mpc/h
SEARCH_RADIUS_FACTOR = 5.0
MASS_THRESHOLD      = 1e9     # M_sun
RHALF_UNIT_FACTOR   = 1e-3    # kpc/h → Mpc/h

MASS_RATIO_MAJOR = 0.25       # >= major merger
MASS_RATIO_MINOR = 0.10       # >= minor merger

## 2 · Data format

`process_galaxies_with_tracks` reads **Caesar HDF5 catalogs directly** — no FITS conversion needed.

### Per-snapshot catalogs
Pass either:
- `caesar_paths` — explicit list of HDF5 paths, ordered to match the track columns
- `sb` + `snaplist` — a `Simulation` object and snapshot list; paths are resolved via `sb.get_caesar_file(snap)`

Required HDF5 datasets (Caesar defaults):

| Dataset path | Description |
|---|---|
| `galaxy_data/pos` | Galaxy centres, shape (N, 3), Mpc/h |
| `galaxy_data/dicts/masses.stellar` | Stellar mass, M☉ |
| `galaxy_data/dicts/radii.stellar_half_mass` | Stellar half-mass radius, kpc/h |
| `galaxy_data/dicts/masses.H2` | Molecular hydrogen mass, M☉ |
| `galaxy_data/dicts/masses.dust` | Dust mass, M☉ |

If your Caesar build uses different internal paths, override the module-level constants before calling:
```python
import simbanator.analysis.mergers as m
m._H5_SMASS = 'galaxy_data/dicts/masses.stellar'   # default shown
```

### Track file (`track_fits_path`)
A small FITS table of shape `(N_galaxies, N_snaps)`.  
Entry `[i, s]` is the **row index** of galaxy *i* in the catalog at snapshot *s*; `-1` means absent.  
Produced by `simbanator.analysis.progenitors.caesar_read_progen` or `read_progen`.

## 3 · Mock data

The cells below build a minimal synthetic dataset so the notebook runs end-to-end without real SIMBA catalogs.  
Skip to **Section 8** to swap in real paths.

In [ ]:
rng = np.random.default_rng(42)

N_GALS  = 10   # tracked galaxies
N_SNAPS = 20   # snapshots
N_CAT   = 200  # galaxies in each snapshot catalog

tmpdir = tempfile.mkdtemp(prefix='simba_merger_')
print('Working directory:', tmpdir)

In [ ]:
# --- per-snapshot Caesar-format HDF5 files -----------------------------
caesar_paths = []

for s in range(N_SNAPS):
    pos   = rng.uniform(0, BOX_SIZE, size=(N_CAT, 3))
    smass = 10 ** rng.uniform(8.5, 12.0, size=N_CAT)   # M_sun
    rhalf = rng.uniform(1, 20, size=N_CAT)              # kpc/h
    h2    = smass * rng.uniform(0.01, 0.5, size=N_CAT)
    dust  = smass * rng.uniform(1e-4, 1e-2, size=N_CAT)

    # Plant a guaranteed close companion for galaxy 0 at snap 5
    if s == 5:
        pos[1]   = pos[0] + rng.uniform(0.001, 0.003, size=3)
        smass[1] = smass[0] * 0.30   # mass ratio 0.30 → major merger

    # Plant a minor-merger companion for galaxy 0 at snap 10
    if s == 10:
        pos[2]   = pos[0] + rng.uniform(0.001, 0.003, size=3)
        smass[2] = smass[0] * 0.12   # mass ratio 0.12 → minor merger

    path = os.path.join(tmpdir, f'snap_{s:03d}.hdf5')
    with h5py.File(path, 'w') as f:
        f.create_dataset('galaxy_data/pos', data=pos)
        f.create_dataset('galaxy_data/dicts/masses.stellar',          data=smass)
        f.create_dataset('galaxy_data/dicts/radii.stellar_half_mass', data=rhalf)
        f.create_dataset('galaxy_data/dicts/masses.H2',               data=h2)
        f.create_dataset('galaxy_data/dicts/masses.dust',             data=dust)
    caesar_paths.append(path)

print(f'Created {len(caesar_paths)} mock Caesar HDF5 catalogs.')

In [ ]:
# --- track FITS file ---------------------------------------------------
# Shape: (N_GALS, N_SNAPS); each entry is a row index into that snapshot's catalog.
# Galaxy 0 always maps to row 0 in every snapshot;
# the others follow with small random drift and occasional gaps (-1).
tracks = np.zeros((N_GALS, N_SNAPS), dtype=int)
for g in range(N_GALS):
    for s in range(N_SNAPS):
        if rng.random() < 0.05:   # 5% chance the galaxy is missing
            tracks[g, s] = -1
        else:
            tracks[g, s] = g      # simple: galaxy g lives at row g

track_path = os.path.join(tmpdir, 'tracks.fits')
cols = [fits.Column(name=f'snap_{s:03d}', format='K', array=tracks[:, s])
        for s in range(N_SNAPS)]
fits.BinTableHDU.from_columns(cols).writeto(track_path, overwrite=True)
print('Track file written:', track_path)
print('Track array shape:', tracks.shape)

## 4 · Run `process_galaxies_with_tracks`

For each snapshot, every tracked galaxy's neighbours within `search_radius_factor × r_half` are recorded as `Progenitor` entries.  
Distances use the **minimum-image convention** (periodic boundary).

In [ ]:
galaxies = process_galaxies_with_tracks(
    track_fits_path      = track_path,
    box_size             = BOX_SIZE,
    caesar_paths         = caesar_paths,
    search_radius_factor = SEARCH_RADIUS_FACTOR,
    mass_threshold       = MASS_THRESHOLD,
    rhalf_unit_factor    = RHALF_UNIT_FACTOR,
)

print(f'\nTracked {len(galaxies)} galaxies.')
total_progenitors = sum(len(g.progenitors) for g in galaxies.values())
print(f'Total merger candidates recorded: {total_progenitors}')

## 5 · Explore the results

The returned dict maps `galaxy_index → Galaxy`.  
Each `Galaxy` holds a dict of `Progenitor` objects keyed by a per-galaxy integer ID.

In [ ]:
gal0 = galaxies[0]
gal0.print_info()

In [ ]:
# Summary of progenitors for each tracked galaxy
print(f'{"Galaxy":>8}  {"# progenitors":>14}  {"snaps with activity":>20}')
print('-' * 50)
for gid, gal in galaxies.items():
    progs = list(gal.progenitors.values())
    snaps = sorted({p.snapshot for p in progs})
    snap_str = ', '.join(map(str, snaps)) if snaps else '—'
    print(f'{gid:>8}  {len(progs):>14}  {snap_str:>20}')

In [ ]:
# Inspect progenitor attributes for galaxy 0
if gal0.progenitors:
    mass_ratios  = gal0.get_attribute_values('mass')
    distances    = gal0.get_attribute_values('distance')
    snap_indices = gal0.get_attribute_values('snapshot')

    print(f'Mass ratios  : min={mass_ratios.min():.3f}  max={mass_ratios.max():.3f}  median={np.median(mass_ratios):.3f}')
    print(f'Distances    : min={distances.min():.4f}  max={distances.max():.4f}  [position units]')
    print(f'Snapshot span: {int(snap_indices.min())} – {int(snap_indices.max())}')
else:
    print('No progenitors found for galaxy 0.')

## 6 · Classify mergers with `analyze_mergers`

Returns two `(n_snaps, n_galaxies)` integer arrays.  

| Mass ratio | Classification |
|---|---|
| ≥ `MASS_RATIO_MAJOR` | **Major** merger |
| ≥ `MASS_RATIO_MINOR` and < major | **Minor** merger |
| < `MASS_RATIO_MINOR` | Ignored |


In [ ]:
major_mergers, minor_mergers = analyze_mergers(
    galaxies,
    array_size       = (N_SNAPS, N_GALS),
    mass_threshold_maj = MASS_RATIO_MAJOR,
    mass_threshold_min = MASS_RATIO_MINOR,
)

print('major_mergers shape:', major_mergers.shape)
print('minor_mergers shape:', minor_mergers.shape)
print(f'Total major events : {major_mergers.sum()}')
print(f'Total minor events : {minor_mergers.sum()}')

In [ ]:
# Per-galaxy totals
print(f'{"Galaxy":>8}  {"Major":>8}  {"Minor":>8}')
print('-' * 28)
for col in range(N_GALS):
    print(f'{col:>8}  {major_mergers[:, col].sum():>8}  {minor_mergers[:, col].sum():>8}')

## 7 · Visualisation

### 7a · Merger rate vs snapshot (sample-averaged)

In [ ]:
snap_indices = np.arange(N_SNAPS)

fig, ax = plt.subplots(figsize=(9, 4))

ax.step(snap_indices, major_mergers.sum(axis=1), where='mid',
        color='firebrick', lw=2, label='Major (ratio ≥ 0.25)')
ax.step(snap_indices, minor_mergers.sum(axis=1), where='mid',
        color='steelblue', lw=2, label='Minor (0.10 ≤ ratio < 0.25)')

ax.set_xlabel('Snapshot index (0 = z=0)')
ax.set_ylabel('Number of merger events')
ax.set_title('Merger count per snapshot (all tracked galaxies)')
ax.legend()
ax.set_xlim(0, N_SNAPS - 1)
plt.tight_layout()
plt.show()

### 7b · Per-galaxy merger history heatmap

In [ ]:
# Combined: major=2, minor=1, none=0
combined = np.zeros_like(major_mergers)
combined[minor_mergers > 0] = 1
combined[major_mergers > 0] = 2

fig, ax = plt.subplots(figsize=(11, 4))

cmap = plt.cm.get_cmap('RdYlGn_r', 3)
im = ax.imshow(combined.T, aspect='auto', origin='lower',
               cmap=cmap, vmin=-0.5, vmax=2.5,
               extent=[-0.5, N_SNAPS - 0.5, -0.5, N_GALS - 0.5])

cbar = fig.colorbar(im, ax=ax, ticks=[0, 1, 2], pad=0.02)
cbar.ax.set_yticklabels(['None', 'Minor', 'Major'])

ax.set_xlabel('Snapshot index')
ax.set_ylabel('Galaxy index')
ax.set_title('Merger history (per galaxy, per snapshot)')
ax.set_yticks(range(N_GALS))
plt.tight_layout()
plt.show()

### 7c · Mass-ratio distribution of all recorded companions

In [ ]:
all_mass_ratios = np.concatenate([
    gal.get_attribute_values('mass')
    for gal in galaxies.values()
    if gal.progenitors
])

fig, ax = plt.subplots(figsize=(7, 4))

bins = np.linspace(0, 1, 30)
ax.hist(all_mass_ratios, bins=bins, color='slategray', edgecolor='white', lw=0.4)

ax.axvline(MASS_RATIO_MAJOR, color='firebrick', ls='--', lw=1.5,
           label=f'Major threshold ({MASS_RATIO_MAJOR})')
ax.axvline(MASS_RATIO_MINOR, color='steelblue', ls='--', lw=1.5,
           label=f'Minor threshold ({MASS_RATIO_MINOR})')

ax.set_xlabel(r'Stellar mass ratio  $m_{\rm sec}\,/\,m_{\rm main}$')
ax.set_ylabel('Count')
ax.set_title('Mass-ratio distribution of merger companions')
ax.legend()
plt.tight_layout()
plt.show()

### 7d · Companion distance distribution

In [ ]:
all_distances = np.concatenate([
    gal.get_attribute_values('distance')
    for gal in galaxies.values()
    if gal.progenitors
])

fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(all_distances, bins=40, color='mediumpurple', edgecolor='white', lw=0.4)
ax.set_xlabel('Separation [position units, Mpc/h]')
ax.set_ylabel('Count')
ax.set_title('Distribution of companion separations')
plt.tight_layout()
plt.show()

### 7e · (Optional) 3-D search sphere for one galaxy at one snapshot

In [ ]:
from simbanator.analysis.mergers import plot_sphere

gal0 = galaxies[0]
progs_snap5 = [p for p in gal0.progenitors.values() if p.snapshot == 5]

if progs_snap5:
    fig = plt.figure(figsize=(7, 6))
    ax3 = fig.add_subplot(111, projection='3d')

    # Main galaxy position
    ax3.scatter(*[gal0.x, gal0.y, gal0.z], s=120, c='gold', zorder=5, label='Main')

    # Companions at snap 5
    xs = [p.x for p in progs_snap5]
    ys = [p.y for p in progs_snap5]
    zs = [p.z for p in progs_snap5]
    ax3.scatter(xs, ys, zs, s=60, c='tomato', label=f'Companions (snap 5)')

    # Approximate search sphere (use gal0.r if set, else default)
    r_search = gal0.r * RHALF_UNIT_FACTOR * SEARCH_RADIUS_FACTOR if gal0.r > 0 else 0.05
    plot_sphere(ax3, gal0.x, gal0.y, gal0.z, r_search, color='steelblue')

    ax3.set_xlabel('x [Mpc/h]')
    ax3.set_ylabel('y [Mpc/h]')
    ax3.set_zlabel('z [Mpc/h]')
    ax3.set_title('Galaxy 0 – companions at snap 5')
    ax3.legend()
    plt.tight_layout()
    plt.show()
else:
    print('No companions at snap 5 for galaxy 0.')

## 8 · Real-data template

Replace the paths below with your actual SIMBA catalog files and re-run from **Section 4** onwards.

In [ ]:
# ── real-data configuration ──────────────────────────────────────────────
# from simbanator.io.simba import Simulation
# from simbanator.analysis.progenitors import caesar_read_progen, read_progen
#
# sim      = Simulation('simba_m100n1024')
# out      = OutputPaths(sim.name)
# BOX_SIZE = 100.0   # Mpc/h
#
# # Ordered list of snapshots (z=0 first, matching track column order)
# snaplist = list(range(151, 5, -1))   # 151 down to 6
#
# # Track file produced by caesar_read_progen or read_progen
# track_path = os.path.join(out.progenitors, 'tracks_QG_z0.fits')
#
# # Option A: pass the Simulation object (paths resolved automatically)
# galaxies = process_galaxies_with_tracks(
#     track_fits_path      = track_path,
#     box_size             = BOX_SIZE,
#     sb                   = sim,
#     snaplist             = snaplist,
#     search_radius_factor = SEARCH_RADIUS_FACTOR,
#     mass_threshold       = MASS_THRESHOLD,
#     rhalf_unit_factor    = RHALF_UNIT_FACTOR,
# )
#
# # Option B: pass explicit HDF5 paths
# # caesar_paths = [sim.get_caesar_file(s) for s in snaplist]
# # galaxies = process_galaxies_with_tracks(
# #     track_fits_path = track_path,
# #     box_size        = BOX_SIZE,
# #     caesar_paths    = caesar_paths,
# # )
#
# major_mergers, minor_mergers = analyze_mergers(
#     galaxies,
#     array_size         = (len(snaplist), len(galaxies)),
#     mass_threshold_maj = MASS_RATIO_MAJOR,
#     mass_threshold_min = MASS_RATIO_MINOR,
# )
print('Uncomment above and fill in real paths.')